# Zoobot
It is a python package that utilies deeplearning models like EfficientNet, DenseNet and ResNet, And unlike generic ImagesNet Models, it was pre-trained specifically on Galaxy Images.

## Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import os
import time

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## Building Confident Labels

In [ ]:
LABELS_PATH = '../data/training_solutions_rev1.csv'
IMG_SIZE = 224  # Zoobot uses 224x224
BATCH_SIZE = 32

labels_df = pd.read_csv(LABELS_PATH)

def assign_class_confident(row):
    if row['Class1.1'] >= 0.7:
        return 'Smooth'
    elif row['Class1.3'] >= 0.7:
        return None
    elif row['Class4.1'] >= 0.7:
        return 'Spiral'
    elif row['Class1.2'] >= 0.7:
        return 'Irregular'
    else:
        return None

labels_df['label'] = labels_df.apply(assign_class_confident, axis=1)
confident_df = labels_df[labels_df['label'].notna()].copy()

confident_df['filepath'] = confident_df['GalaxyID'].apply(
    lambda gid: os.path.join(IMG_DIR, f"{gid}.jpg")
)

le = LabelEncoder()
confident_df['label_encoded'] = le.fit_transform(confident_df['label'])

print("Classes:", le.classes_)
print(confident_df['label'].value_counts())
print(f"Total confident galaxies: {len(confident_df)}")

## Train/val/test Splitting

In [ ]:
filepaths = confident_df['filepath'].values
labels = confident_df['label_encoded'].values

X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    filepaths, labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(f"Train: {len(X_train_paths)}")
print(f"Val:   {len(X_val_paths)}")
print(f"Test:  {len(X_test_paths)}")

## Class Weights

In [ ]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weight_tensor = torch.FloatTensor(class_weights)
print("Class weights:", dict(zip(classes, class_weights)))

## Pytorch Dataset Class

In [ ]:
class GalaxyDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

## Transforms and Dataloaders

In [ ]:
# training transforms — augmentation + normalization
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# val/test transforms — resize and normalize only, no augmentation
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = GalaxyDataset(X_train_paths, y_train, transform=train_transform)
val_dataset = GalaxyDataset(X_val_paths, y_val, transform=val_transform)
test_dataset = GalaxyDataset(X_test_paths, y_test, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# sanity check
images, labels_batch = next(iter(train_loader))
print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels_batch.shape}")

## Loading Zoobot Pretrained Model

In [ ]:
import inspect
from zoobot.pytorch.training.finetune import load_pretrained_zoobot
print(inspect.signature(load_pretrained_zoobot))
print(inspect.getdoc(load_pretrained_zoobot))

In [ ]:
from huggingface_hub import list_repo_files

repo_id = 'mwalmsley/zoobot-encoder-efficientnet_b0'
files = list(list_repo_files(repo_id))
print(f"Files in {repo_id}:")
for f in files:
    print(f" - {f}")

In [ ]:
from huggingface_hub import hf_hub_download
import timm
import torch
import torch.nn as nn

# download actual Zoobot galaxy-pretrained weights
print("Downloading Zoobot galaxy-pretrained weights...")
weights_path = hf_hub_download(
    repo_id='mwalmsley/zoobot-encoder-efficientnet_b0',
    filename='pytorch_model.bin'
)
print(f"Downloaded to: {weights_path}")

# create EfficientNetB0 encoder using timm
encoder = timm.create_model(
    'efficientnet_b0',
    pretrained=False,  # don't load ImageNet weights — we'll load Zoobot weights instead
    num_classes=0      # feature extractor only, no classification head
)

# load Zoobot's galaxy-pretrained weights into the encoder
state_dict = torch.load(weights_path, map_location='cpu')
print(f"Keys in state dict (first 5): {list(state_dict.keys())[:5]}")
print(f"Total keys: {len(state_dict.keys())}")

In [ ]:
from huggingface_hub import hf_hub_download
import timm
import torch
import torch.nn as nn

# download actual Zoobot galaxy-pretrained weights
print("Downloading Zoobot galaxy-pretrained weights...")
weights_path = hf_hub_download(
    repo_id='mwalmsley/zoobot-encoder-efficientnet_b0',
    filename='pytorch_model.bin'
)
print(f"Downloaded to: {weights_path}")

# create EfficientNetB0 encoder using timm
encoder = timm.create_model(
    'efficientnet_b0',
    pretrained=False,  # don't load ImageNet weights
    num_classes=0      # feature extractor only
)

# load Zoobot's galaxy-pretrained weights
state_dict = torch.load(weights_path, map_location='cpu')
missing_keys, unexpected_keys = encoder.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {len(missing_keys)}")
print(f"Unexpected keys: {len(unexpected_keys)}")
if missing_keys:
    print("Missing:", missing_keys[:5])
if unexpected_keys:
    print("Unexpected:", unexpected_keys[:5])
print("Zoobot galaxy-pretrained weights loaded successfully.")

# build classifier on top of Zoobot encoder
class ZoobotClassifier(nn.Module):
    def __init__(self, encoder, num_classes=3):
        super(ZoobotClassifier, self).__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.encoder(x)
        # handle both 4D (spatial) and 2D (already pooled) outputs
        if len(features.shape) == 4:
            features = features.mean(dim=[2, 3])
            return self.classifier[1:](features)  # skip AdaptiveAvgPool2d
        return self.classifier(features)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ZoobotClassifier(encoder, num_classes=3).to(device)
print(f"Model on: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Ready to train with actual galaxy-pretrained weights.")

## Training Utilites

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

## Phase 1: Training Classifier Head Only (Freeze Encoder)

In [ ]:
class ZoobotClassifier(nn.Module):
    def __init__(self, encoder, num_classes=3):
        super(ZoobotClassifier, self).__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.encoder(x)
        # timm with num_classes=0 already returns
        # globally pooled 1D vector (batch, 1280)
        # no need for AdaptiveAvgPool2d or manual pooling
        return self.classifier(features)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ZoobotClassifier(encoder, num_classes=3).to(device)
print(f"Model on: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# quick shape check before training
test_input = torch.randn(2, 3, 224, 224).to(device)
test_output = model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print("Forward pass works correctly.")

In [ ]:
# freeze encoder
for param in model.encoder.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss(
    weight=class_weight_tensor.to(device)
)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

print("Phase 1: Training classifier head only (encoder frozen)...")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

history_phase1 = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 3

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history_phase1['train_acc'].append(train_acc)
    history_phase1['val_acc'].append(val_acc)
    history_phase1['train_loss'].append(train_loss)
    history_phase1['val_loss'].append(val_loss)

    print(f"Epoch {epoch+1}/10 — train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, "
          f"val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")

    # manual early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), '/content/best_phase1.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

# restore best weights
model.load_state_dict(torch.load('/content/best_phase1.pt'))
print("Phase 1 complete. Best weights restored.")

## Phase 2: Unfreeze Encoder and Fine-tune

In [ ]:
# unfreeze encoder
for param in model.encoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-5
)

print("Phase 2: Fine-tuning full model (encoder unfrozen)...")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

history_phase2 = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(15):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history_phase2['train_acc'].append(train_acc)
    history_phase2['val_acc'].append(val_acc)
    history_phase2['train_loss'].append(train_loss)
    history_phase2['val_loss'].append(val_loss)

    print(f"Epoch {epoch+1}/15 — train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, "
          f"val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), '/content/best_phase2.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load('/content/best_phase2.pt'))
print("Phase 2 complete. Best weights restored.")

## Evaluating On Test Set

In [ ]:
criterion_eval = nn.CrossEntropyLoss()
test_loss, test_accuracy = evaluate(model, test_loader, criterion_eval, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Plotting Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_acc = history_phase1['train_acc'] + history_phase2['train_acc']
val_acc = history_phase1['val_acc'] + history_phase2['val_acc']
train_loss = history_phase1['train_loss'] + history_phase2['train_loss']
val_loss = history_phase1['val_loss'] + history_phase2['val_loss']

phase1_epochs = len(history_phase1['train_acc'])

axes[0].plot(train_acc, label='Train Accuracy')
axes[0].plot(val_acc, label='Val Accuracy')
axes[0].axvline(x=phase1_epochs, color='gray', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Zoobot Accuracy (Confident Dataset)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(train_loss, label='Train Loss')
axes[1].plot(val_loss, label='Val Loss')
axes[1].axvline(x=phase1_epochs, color='gray', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Zoobot Loss (Confident Dataset)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/zoobot_curves.png')
plt.show()

## Saving Model

In [ ]:
torch.save(
    model.state_dict(),
    '../models/efficientnetb3_confident.keras'
print("Model saved to Drive.")

from google.colab import files
files.download('../reports/figures/zoobot_curves.png')